# model_12 — HİBRİT: Gated DeltaNet-2 ×6 + tam dikkat ×2

**Soru:** 2026'nın güncel mimarisi, bizim dilimizde `model_11` ile
**aynı sonucu** veriyor mu?

Artımlı bir kol değil — doğru kurulmuş **ikinci referans**. Karıştırıcı +
optimizer + precision birlikte değişiyor, yani sayılar farklı çıkarsa
*neden* farklı olduğunu söyleyemeyeceğiz. Kabul edildi.

**Sabit kalan tek şey veri ve sınav** (ölçme izi `44e6262e37f3`, graf izi
`3cd9a2575e47`, korpus izi `977337b5bc33`). Bu atfetme kuralı değil,
ölçümün kendisi: sınav da değişirse elde kıyaslanabilir iki sayı değil,
ilgisiz iki sayı kalır.

```
blok   1  2  3  4  5  6  7  8
       G  G  G  A  G  G  G  A        3:1
G = Gated DeltaNet-2 (2605.22791)    A = MHA + RoPE + QK-norm
her blokta SwiGLU FFN (dff=704) -- model_11 ile AYNI, bilerek
optimizer Muon   precision bf16   Lookahead KAPALI
```

Hücre künyesi ilk satırda: `# <n> <ADIM> | CPU/GPU | tekrar: GUVENLI/degil`

In [ ]:
# 0 MODEL ADI VE YOLLAR  |  CPU  |  tekrar: GUVENLI
MODEL = "model_12"
import time
DEPO = "https://github.com/sekerahmet/sekerai.git"
KOD  = "/content/kod"
EV   = f"/content/drive/MyDrive/{MODEL}"
# Log adi BURADA uretilmez -- baslatan hucre kendi damgasini basar,
# yoksa ikinci kosu oncekinin logunu "w" ile sifirlar (15 Eylul'de oldu).
print(MODEL, "->", EV)

In [ ]:
# 1 GPU VAR MI  |  GPU'yu SORAR  |  tekrar: GUVENLI
import torch
assert torch.cuda.is_available(), "GPU YOK -- Calisma zamani > Turu degistir"
_p = torch.cuda.get_device_properties(0)
_bos = torch.cuda.mem_get_info()[0] / 1e9
print(f"{_p.name}   toplam {_p.total_memory/1e9:.1f} GB   bos {_bos:.1f} GB")
assert _bos > 3.0, f"GPU'da sadece {_bos:.1f} GB bos"

In [ ]:
# 2 DRIVE  |  CPU  |  tekrar: GUVENLI
from google.colab import drive
import os
drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "drive.mount CALISMADI"
os.makedirs(f"{EV}/log", exist_ok=True)
with open(f"{EV}/.yazma_denemesi", "w") as f:
    f.write("ok")
os.remove(f"{EV}/.yazma_denemesi")
print("hazir ve YAZILABILIR:", EV)
print("  tohum klasorleri:",
      sorted(d for d in os.listdir(EV) if d.startswith("t")) or "(yok)")

In [ ]:
# 3 KODU CEK + KILIT TESTI  |  CPU  |  tekrar: KOSU YOKKEN
import subprocess, os, glob, importlib, sys
subprocess.run(["rm", "-rf", KOD], check=True)
subprocess.run(["git", "clone", "--depth", "1", DEPO, KOD], check=True)
COMMIT = subprocess.run(["git", "-C", KOD, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("commit:", COMMIT, "|",
      subprocess.run(["git", "-C", KOD, "log", "-1", "--format=%s"],
                     capture_output=True, text=True).stdout.strip())
AILE = f"{KOD}/deneme2/{MODEL}"
assert os.path.isdir(AILE), AILE
PENCERE = f"{AILE}/pencere_12.py"
assert os.path.exists(PENCERE), PENCERE

# IMPORT ONBELLEGI: `rm -rf` + klon kodu tazeler ama sys.modules ESKI
# modul nesnesini tutar -- yeni kodu klonlayip eski kodla kosarsin.
_atilan = [n for n, m in list(sys.modules.items())
           if getattr(m, "__file__", None) and str(m.__file__).startswith(KOD)]
for n in _atilan:
    del sys.modules[n]
importlib.invalidate_caches()
if _atilan:
    print("import onbellegi temizlendi:", len(_atilan), "modul")

# KILIT TESTI -- duserse EGITIM BASLAMAMALI.
print()
print("=" * 72)
_r = subprocess.run([sys.executable, f"{AILE}/test_12.py"], cwd=AILE,
                    capture_output=True, text=True,
                    env={**os.environ, "AYRINTI": "1",
                         "KOSU_KOK": EV.rsplit("/", 1)[0]})
print(_r.stdout.rstrip() or "(cikti YOK -- test kosmamis olabilir!)")
if _r.stderr.strip():
    print("stderr:", _r.stderr.rstrip()[-2000:])
print("=" * 72)
assert _r.returncode == 0, "test_12 DUSTU -- EGITIM BASLATMA."
print()

sys.path.insert(0, AILE)
M_12 = importlib.import_module(MODEL)
M = M_12
MOTOR = M_12.M
assert M.AYAR.ad == MODEL
assert M.__file__.startswith(KOD), f"{MODEL} {KOD} disindan geldi"

# --- MIMARI: kolun TANIMI ---------------------------------------------
assert M.AYAR.dongu == 1, "DONGU YOK"
assert M.AYAR.l == 8, "8 blok"
assert M.AYAR.d == 256, "d"
assert M.AYAR.nh == 4, "head_dim 64"
assert M.AYAR.dff == 704, "SwiGLU 8/3*d"
assert M.AYAR.karisim == "GGGAGGGA", "3:1 hibrit, dikkat 4. ve 8. blokta"
assert M.AYAR.gdn_v_kat == 2, "Hv = 2*nh (Qwen3-Next)"
assert M_12.G_TABAN == -5.0, "adim basina sonum tabani, alpha >= exp(-5)"
assert M_12._gdn2_parcali.__defaults__[0] == 16, "chunk 16 -- OLCUMLE secildi"
assert M.AYAR.dar_alfa == 0.0, "Phi DARBOGAZI YOK"
assert M.AYAR.kopru_kayip == 0.0, "YARDIMCI KAYIP YOK"
_bl = "".join("A" if isinstance(b.mix, M_12.Dikkat) else "G"
              for b in M_12.ModelHibrit(M.AYAR, 64).bloklar)
assert _bl == M.AYAR.karisim, f"desen {_bl} != {M.AYAR.karisim}"
print("mimari:", _bl, " GDN-2 x", _bl.count("G"), " dikkat x", _bl.count("A"))

# --- EGITIM ------------------------------------------------------------
assert M.AYAR.optim == "muon", "Muon -- 2026 uretim tarifi"
assert M.AYAR.bf16 == True, "bf16 -- GDN-2 durumu fp32"
assert M.AYAR.derle == True, "torch.compile 1.83x"
assert M.AYAR.wd == 0.5, "model_a3 recetesi"
assert M.AYAR.sabit_lr == True, "cosine KAPALI"
assert M.AYAR.lr == 1e-3, "Pythia-70m mertebesi"
assert M.AYAR.batch == 32, "OLCULMUS OPTIMUM"
assert M.AYAR.adim == 20000, "CLAUDE.md kural 1 -- ILK SINIR"
assert M.AYAR.isinma == 2000, "kosunun %10'u"
assert M.AYAR.olc_her == 2000, "10 olcum noktasi"
assert M.AYAR.ort_bas == 0, "LOOKAHEAD KAPALI"

# --- VERI VE SINAV: model_11 ile AYNI KALMALI --------------------------
assert M.AYAR.veri_ad == "veri_12", "kendi veri modulu"
assert M.AYAR.jeton_ad == "tam", "varlik = kelime dizisi"
assert M.AYAR.ek_kip == "tr2", "gercek Turkce allomorflar"
assert M.AYAR.t_len == 512, "egitim penceresi"
assert M.AYAR.kopya == 40, "varlik basina belge"
assert M.AYAR.tetik == 16, "biyografi onegi orani 2/5"
assert M.AYAR.zincir_pay == 0.20, "zincir AZINLIK"
assert M.AYAR.n3 == 10000, "uc adimli zincir havuzu"
assert M.AYAR.ret_pay == 0.05, "reddetme payi"
assert M.AYAR.ret_tut == 0.2, "reddetmenin bir kismi SINAVA"
assert M.AYAR.ood_pay == 0.05, "ood bolmesi"
assert M.AYAR.bicim == 1, "satir tablosu KAPALI"

import veri_12 as _V12, ayar_12 as _AY12
assert _V12.graf_izi(_V12.kur(0)) == _V12.IZ == "3cd9a2575e47", "graf KAYMIS"
_G12 = _V12.kur(0)
print(f"veri_12  graf izi {_V12.IZ}   {sum(_G12['n'].values())} varlik  "
      f"{len(_G12['olgu'])} olgu  |R| {len(_V12.ILISKI)}")
print(f"butce: {M.AYAR.adim:,} adim = {_AY12.EPOK:.1f} epok   "
      f"korpus {_AY12.KORPUS_JETON:,} jeton")
print("ayar:", M.AYAR)

In [ ]:
# 4 BASLAT  |  GPU (alt surec)  |  tekrar: HAYIR -- yeni kosu baslatir
# --- GPU KAPISI (CLAUDE.md kural 2)
import torch
assert torch.cuda.is_available(), "GPU YOK"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")

TOHUMLAR = [0]
# ILK SINIR 20.000 (kural 1). Uzatma AYRI bir karar ve SURDURMEDIR:
#   ADIM = 60000 ; SURDUR = True   ->  kos_12.py --adim 60000 --surdur
# model_11 tam bu yoldan gitti; ayni yolu izleyince IKI kiyas noktasi
# olur (20k ve 60k). sabit_lr=True oldugu icin uzatma temiz: uzatilan
# adimlar bastan uzun bir kosunun gorecegi LR'nin AYNISINI gorur.
ADIM   = None    # None = ayar_12'deki 20.000
SURDUR = False
USTUNE = False   # True = dolu t<N>'i t<N>_eski_<zaman>/'a TASIR (silmez)

if "p" in globals() and p.poll() is None:
    raise SystemExit(f"ZATEN KOSUYOR (PID {p.pid}). Once DURDUR hucresi.")

_arg = [sys.executable, "-u", f"{AILE}/kos_12.py",
        "--ev", EV, "--commit", COMMIT,
        "--tohum", *[str(t) for t in TOHUMLAR]]
if ADIM:
    _arg += ["--adim", str(ADIM)]
if SURDUR:
    _arg += ["--surdur"]
if USTUNE:
    _arg += ["--ustune"]
LOG = f"{EV}/log/kos_{time.strftime('%Y%m%d_%H%M%S')}.txt"
p = subprocess.Popen(_arg, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print("PID", p.pid, " tohum", TOHUMLAR, " -> log:", LOG)
print("Dolu bir tohum klasoru varsa kosu REDDEDILIR -- hicbir sey ezilmez.")

In [ ]:
# 5 ILERLEME (ham log)  |  CPU  |  tekrar: GUVENLI
import glob, subprocess
_l = sorted(glob.glob(f"{EV}/log/kos_*.txt"))
assert _l, f"log yok: {EV}/log/"
# `p` cekirdek yeniden baslayinca kaybolur -- tam o an bu hucre gerekir.
if "p" in globals():
    _d = p.poll()
    print("KOSUYOR" if _d is None else f"BITTI/OLDU (cikis {_d})",
          "| PID", p.pid)
else:
    _ps = subprocess.run(
        ["bash", "-lc", "ps -eo pid,etime,cmd | grep kos_12.py | grep -v grep"],
        capture_output=True, text=True).stdout.strip()
    print("`p` YOK -- cekirdek yeniden baslamis.")
    print("   SUREC YASIYOR:\n   " + _ps if _ps else
          "   kos_12.py de YOK -> kosu OLDU. 2->3 kos, sonra SURDUR=True.")
print("log:", _l[-1].split("/")[-1], f"({len(_l)} log)")
print("-" * 78)
print(open(_l[-1]).read()[-4000:])

In [ ]:
# 6 RAPOR (canli durum)  |  CPU  |  tekrar: GUVENLI
import json, glob, os, statistics

# AYRIM TEK SORUDA: cevap korpusta yaziyor mu?
#   EVET -> HAFIZA (bulup getirecek)    HAYIR -> CIKARIM (birlestirecek)
IZ_12 = "44e6262e37f3"      # test_12 §7c bunu gercek olcme iziyle sinar
KIYAS = {
    "model_11 t0 @20.000": dict(olgu=0.9353, zincir=0.0853,
                                gorulmemis=0.0189, yabanci=0.0137),
    "model_11 t0 @54.000": dict(olgu=0.9887, zincir=0.0827,
                                gorulmemis=0.0167, yabanci=0.0153),
}
# ^ AYNI SINAV (iz 44e6262e37f3), AYNI veri. Fark ALINIR.
#   model_11 t0 @60.000 kosu bitince buraya eklenecek.

AD = (("wug", "ek kurali", "DIL", "UYDURMA kokte ek uyumu -- sans 0.25"),
      ("one", "olgu", "HAFIZA", "TEK bir sey ogretildi"),
      ("seen", "zincir", "HAFIZA", "BILESIK bir sey ogretildi"),
      ("comp", "gorulmemis", "CIKARIM", "o zincir korpusta HIC gecmedi"),
      ("comp_kisayol", "cikarimsiz", "CIKARIM", "birlestirmedi, tek adim"),
      ("ent", "yabanci", "CIKARIM", "varlik hic zincir basi olmamis"),
      ("dogru_ret", "ret", "DURUSTLUK", "cevapsiz soruya 'yok' dedi mi"),
      ("kacamak", "kacamak", "DURUSTLUK", "BILDIGINE 'yok' dedi mi -- KOTU"),
      ("ent_yok", "birim", "OLCUM", "kisayol IMKANSIZ -- 0 OLMALI"),
      ("ood", "ood", "OLCUM", "80 ornek -- GURULTULU"))
W = 12
_h = lambda x, n=4: (f"{'---':>{W}}" if x is None else f"{x:>{W}.{n}f}")

for kl in sorted(k for k in glob.glob(f"{EV}/t*") if os.path.isdir(k)):
    _ad = os.path.basename(kl)
    if "_eski_" in _ad:
        continue
    eg = glob.glob(f"{kl}/egri_*.json")
    if not eg:
        print(f"{_ad}: egri YOK")
        continue
    ky = glob.glob(f"{kl}/kosu_t*.json")
    k = json.load(open(ky[0])) if ky else {}
    e = json.load(open(eg[0]))
    # Izi tutmayan kosu ATLANIR, PATLAMAZ -- bayat bir klasor calisan
    # kosunun raporunu goturmesin (19 Eylul'de oldu).
    if k.get("olcme_izi") != IZ_12:
        print(f"{_ad}: BASKA SINAV (iz {k.get('olcme_izi','?')}) -- ATLANDI")
        continue
    _var = set().union(*(set(r) for r in e))
    _b = [x for x in AD if x[0] in _var]
    _yk = [c for c in _var if c.endswith("_yakin")]
    s, _dur = e[-1], k.get("durum", "?")
    _son = s.get("adim", 0)
    _ay = glob.glob(f"{kl}/ayar_t*.json")
    _hedef = max(_son, (json.load(open(_ay[0])).get("adim") if _ay else 0) or 0)
    _bpc = [r["bpc"] for r in e if r.get("bpc") is not None]

    print("=" * 108)
    print(f" {_ad}   adim {_son:,}/{_hedef:,}   durum {_dur}"
          f"   commit {k.get('commit','?')}   {k.get('gpu','')}")
    print("=" * 108)
    _eksik = []
    if len(_bpc) >= 2:
        _dus = _bpc[-1] < _bpc[-2] - 1e-4
        print(f"   {'DIL':<22}{'bpc ' + format(_bpc[-1], '.3f'):<14}"
              + ("HALA DUSUYOR -- doymadi" if _dus else "DUZLESTI"))
        if _dus:
            _eksik.append("bpc hala dusuyor")
    for _g, _t, _c, _e2 in (("DIL", "ek kurali", "wug", 0.90),
                            ("HAFIZA", "olgu", "one", 0.98),
                            ("HAFIZA", "zincir", "seen", 0.95),
                            ("CIKARIM", "gorulmemis", "comp", 0.50)):
        _v = s.get(_c)
        if _v is None:
            continue
        print(f"   {_g + '/' + _t:<22}{format(_v, '.4f'):<14}"
              + ("GECTI" if _v >= _e2 else f"YETMEDI   esik {_e2}"))
        if _v < _e2:
            _eksik.append(f"{_g}/{_t} {_v:.4f} < {_e2}")
    if s.get("dogru_ret") is not None:
        _kc = s.get("kacamak", 0.0)
        print(f"   {'DURUSTLUK/ret':<22}{format(s['dogru_ret'], '.4f'):<14}"
              f"ad {s.get('ret_ad', 0):.4f}  cift {s.get('ret_cift', 0):.4f}")
        print(f"   {'DURUSTLUK/kacamak':<22}{format(_kc, '.4f'):<14}"
              + ("!! YUKSEK -- ret TEK BASINA OKUNMAZ" if _kc > 0.05
                 else "dusuk -- ret okunabilir"))
    print(f"   {'TUTARLILIK':<22}{'konus_12 --panel':<14}(13. hucre)")
    print()
    if s.get("ent_yok_kisayol") and abs(s["ent_yok_kisayol"]) >= 1e-9:
        print("   !! BIRIM TESTI DUSTU -- HICBIR SAYI OKUNMAZ.")
    elif _dur == "KOSUYOR":
        print(f"   KOSU DEVAM EDIYOR ({_son:,}/{_hedef:,}) -- HUKUM YOK.")
        print("   Hukum kosu bitince `pencere_12` ile verilir.")
    elif _eksik:
        print("   OLGUN MU?  HAYIR:", "; ".join(_eksik))
    else:
        print("   OLGUN MU?  DIL / HAFIZA / CIKARIM alanlarinda EVET.")

    print()
    print("   " + f"{'adim':>7}{'bpc':>{W}}"
          + "".join(f"{t:>{W}}" for _a, t, _g, _n in _b) + f"{'dk':>6}")
    for r in e:
        print("   " + f"{r['adim']:>7d}" + _h(r.get("bpc"), 3)
              + "".join(_h(r.get(a)) for a, _t, _g, _n in _b)
              + f"{r['sn']/60:>6.0f}")
    print()
    for a, t, _g, n in _b:
        print(f"   {t:<13}{n:<48}{a}")
    d = [(b2["sn"] - a2["sn"]) / (b2["adim"] - a2["adim"]) * 1000
         for a2, b2 in zip(e, e[1:])
         if b2["sn"] > a2["sn"] and b2["adim"] > a2["adim"]]
    if d:
        print(f"\n   HIZ ortanca {statistics.median(d):.1f} ms/adim")
    print("   KIYAS -- AYNI SINAV (iz %s), fark ALINIR:" % IZ_12)
    for _n, _d in KIYAS.items():
        print(f"      {_n:<24}"
              + "  ".join(f"{a}{b:.4f}" for a, b in _d.items()))

In [ ]:
# 7 pencere_12 -- BIRINCIL OKUMA  |  GPU  |  ANCAK KOSU BITINCE
# --- GPU KAPISI (CLAUDE.md kural 2)
import torch
assert torch.cuda.is_available(), "GPU YOK"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")

# N anlik goruntunun AGIRLIK ORTALAMASI, tek model olculur.
# Egri degeri ile pencere degeri AYNI SEY DEGILDIR.
TOHUM = TOHUMLAR[0]
!python {PENCERE} {EV}/t{TOHUM} --genislik 5

In [ ]:
# 8 DURDUR  |  CPU  |  tekrar: KOSUYU OLDURUR -- bastaki # bilerek duruyor
# Anlik goruntuler Drive'da kalir; surdurme paketi her olcum noktasinda
# yazilir, 4. hucrede SURDUR=True ile devam edilir.
# p.kill()

In [ ]:
# 9 BUTUN ANALIZLER -- ARKA PLANDA  |  GPU  |  tekrar: GUVENLI
# --- GPU KAPISI (CLAUDE.md kural 2)
import torch, os, sys, time, subprocess
assert torch.cuda.is_available(), "GPU YOK"
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, f"GPU'da sadece {_bos:.1f} GB bos"
print(f"GPU kapisi GECTI: {torch.cuda.get_device_name(0)}  bos {_bos:.1f} GB")

# Dort arac AYRI hucrelerdeydi ve sirayla elle kosuluyordu; ucu hic
# kosulmadi. Tek Popen, tek log, hucre HEMEN doner.
MODEL = globals().get("MODEL", "model_12")
EV = globals().get("EV", f"/content/drive/MyDrive/{MODEL}")
AILE = globals().get("AILE", f"/content/kod/deneme2/{MODEL}")
T0 = globals().get("TOHUMLAR", [0])[0]
KL = f"{EV}/t{T0}"
assert os.path.isdir(KL), KL
_ARAC = [("tani_12      ARIZA SEKLI",   "tani_12.py", []),
         ("durustluk_12 RET + KACAMAK", "durustluk_12.py", ["--ornek", "12"]),
         ("asama1_12    KOPRU SONDASI", "asama1_12.py",
          ["--bolme", "comp,ent,ent_yok,ood"]),
         ("konus_12     TUTARLILIK",    "konus_12.py", ["--panel"])]
ANALIZ = f"{EV}/log/analiz_{time.strftime('%Y%m%d_%H%M%S')}.txt"
_sh = " ; ".join(
    f"echo '===== {ad} =====' ; {sys.executable} -u {os.path.join(AILE, d)}"
    f" {KL} {' '.join(x)} 2>&1 || echo '!! {ad} DUSTU'"
    for ad, d, x in _ARAC) + " ; echo '===== HEPSI BITTI ====='"
pa = subprocess.Popen(["bash", "-lc", _sh], cwd=AILE,
                      stdout=open(ANALIZ, "w"), stderr=subprocess.STDOUT)
print("PID", pa.pid, "->", ANALIZ)

In [ ]:
# 10 ANALIZ ILERLEMESI  |  CPU  |  tekrar: GUVENLI
import glob, subprocess
_l = sorted(glob.glob(f"{EV}/log/analiz_*.txt"))
assert _l, f"analiz logu yok: {EV}/log/"
if "pa" in globals():
    _d = pa.poll()
    print("KOSUYOR" if _d is None else f"BITTI (cikis {_d})", "| PID", pa.pid)
else:
    _ps = subprocess.run(
        ["bash", "-lc", "ps -eo pid,etime,cmd | grep _12.py | grep -v grep"],
        capture_output=True, text=True).stdout.strip()
    print("`pa` YOK -- cekirdek yeniden baslamis.")
    print("   SUREC YASIYOR:\n   " + _ps if _ps else "   surec de YOK.")
_s = open(_l[-1]).read()
print("log:", _l[-1].split("/")[-1], f"({len(_s):,} karakter)")
print("BITEN:", [x for x in _s.split("\n") if x.startswith("=====")])
print("-" * 78)
print(_s[-6000:])